In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.1 生成脚本: Single-Voxel Signature Concept
--------------------------------------------------
功能：
1. 读取 .mat 数据 (341 维)。
2. 自动或手动选择一个脑内体素。
3. 绘制 "Voxel Signature" 概念图：
   - Top: 5张代表性模态切片，带选点标记。
   - Bottom: 341维特征向量曲线 (Z-score 归一化)，带分区背景。

适配数据集: 0-340 (共341通道)
0-14: QTI Metrics
15-224: Raw QTI
225-228: CEST Params
229-340: Z-Spectra
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from pathlib import Path
import h5py

# ================= 配置区域 =================
# 输入文件路径 (请修改此处)
DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/your_patient_data.mat"

# 选点配置 (None 表示自动选择 mask 中心)
VOXEL_COORDS = None  # 格式: (x, y, z) 0-based

# 输出路径
OUTPUT_FIG = "Figure_1_1_Concept.png"
DPI = 300

# 模态区间定义 (0-based, 左闭右闭)
SECTIONS = [
    {"name": "QTI Metrics", "start": 0,   "end": 14,  "color": "#FFEEEE"}, # 红淡
    {"name": "Raw QTI",     "start": 15,  "end": 224, "color": "#EEF5FF"}, # 蓝淡
    {"name": "CEST Fits",   "start": 225, "end": 228, "color": "#EEFFEE"}, # 绿淡
    {"name": "Z-Spectra",   "start": 229, "end": 340, "color": "#FDFEEE"}, # 黄淡
]

# 代表性切片通道 (用于上方展示)
# 建议选择: FA(6), 一张RawQTI(100), NOE(227), Z-spec(250), M0(229)
SAMPLE_CHANNELS = [
    {"idx": 6,   "label": "FA (QTI Metric)"},
    {"idx": 100, "label": "Raw QTI (b-tensor)"},
    {"idx": 227, "label": "NOE (CEST Param)"},
    {"idx": 250, "label": "Raw Z-Spectrum"},
    {"idx": 229, "label": "M0 / Ref"},
]

# ===========================================

def load_data(mat_path):
    """读取数据 (复用逻辑)"""
    print(f"Loading: {mat_path} ...")
    with h5py.File(mat_path, 'r') as f:
        data = f['data'][:]
        # 维度转置处理 (341, X, Y, Z) -> (X, Y, Z, 341)
        if data.shape[0] == 341 or data.shape[0] == 351:
            data = np.moveaxis(data, 0, -1)
        
        region_mask = f['region_mask'][:].astype(bool)
        
    print(f"Data Shape: {data.shape}")
    return data, region_mask

def normalize_image(img):
    """单张图像 1-99% 归一化 (用于显示)"""
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)

def select_voxel(mask):
    """自动选择 Mask 重心附近的体素"""
    indices = np.argwhere(mask)
    if len(indices) == 0:
        raise ValueError("Mask is empty!")
    center = indices.mean(axis=0).astype(int)
    # 找离重心最近的 True 点
    dists = np.sum((indices - center)**2, axis=1)
    best_idx = indices[np.argmin(dists)]
    return tuple(best_idx)

def plot_figure_1_1(data, voxel, output_path):
    """绘制论文图 1.1"""
    vx, vy, vz = voxel
    print(f"Selected Voxel: x={vx}, y={vy}, z={vz}")
    
    # 提取特征向量
    spectrum = data[vx, vy, vz, :].astype(np.float32)
    # Z-score 归一化 (符合论文 Single-voxel signature 风格)
    spectrum_norm = (spectrum - np.mean(spectrum)) / (np.std(spectrum) + 1e-8)
    
    # 设置画板
    fig = plt.figure(figsize=(14, 8), facecolor='white')
    gs = gridspec.GridSpec(2, 5, height_ratios=[1, 1.5], hspace=0.3, wspace=0.1)
    
    # --- Top: Representative Slices ---
    for i, item in enumerate(SAMPLE_CHANNELS):
        ax = fig.add_subplot(gs[0, i])
        ch_idx = item['idx']
        
        # 提取切片 (Axial)
        if ch_idx < data.shape[-1]:
            slc = data[:, :, vz, ch_idx]
            slc_norm = normalize_image(slc)
            
            # 旋转90度以符合常规解剖视角 (视情况调整)
            slc_show = np.rot90(slc_norm)
            # 坐标转换 (旋转后的坐标)
            # 原图 (x, y) -> rot90 -> (y, H-x-1)
            # 注意: matplotlib imshow 的 x 是列, y 是行
            show_x = vy
            show_y = slc.shape[0] - 1 - vx 
            
            ax.imshow(slc_show, cmap='gray')
            
            # 绘制十字准星
            ax.scatter(show_x, show_y, c='red', s=40, marker='+', linewidth=1.5)
            # 画个圈
            circ = patches.Circle((show_x, show_y), radius=5, edgecolor='red', facecolor='none', lw=0.8)
            ax.add_patch(circ)
            
            ax.set_title(f"{item['label']}\n(Ch {ch_idx+1})", fontsize=10, fontweight='bold')
        
        ax.axis('off')

    # --- Bottom: Voxel Signature ---
    ax_main = fig.add_subplot(gs[1, :])
    
    # 绘制背景分区
    for sec in SECTIONS:
        # 确保不越界
        s = max(0, sec['start'])
        e = min(len(spectrum), sec['end'])
        if s < e:
            ax_main.axvspan(s, e, color=sec['color'], alpha=1.0, zorder=0)
            # 添加分区文字
            mid = (s + e) / 2
            ax_main.text(mid, np.max(spectrum_norm)*1.05, sec['name'], 
                         ha='center', va='bottom', fontsize=9, fontweight='bold', color='#333333')

    # 绘制曲线
    ax_main.plot(spectrum_norm, color='#2c3e50', linewidth=1.2, zorder=10)
    
    # 样式调整
    ax_main.set_xlim(0, 341)
    # y轴范围稍微放宽一点放文字
    ymin, ymax = np.min(spectrum_norm), np.max(spectrum_norm)
    ax_main.set_ylim(ymin, ymax + (ymax-ymin)*0.15)
    
    ax_main.set_xlabel("Channel Index (0-340)", fontsize=11)
    ax_main.set_ylabel("Normalized Intensity (Z-score)", fontsize=11)
    ax_main.set_title(f"Single-Voxel Signature (Voxel: {vx}, {vy}, {vz})", fontsize=13, pad=20)
    
    # 精细化刻度
    ax_main.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)

    # 保存
    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()

def main():
    # 1. 加载
    data_4d, mask = load_data(DATA_PATH)
    
    # 2. 检查维度 (兼容 341 或 351)
    C = data_4d.shape[-1]
    print(f"Detected Channels: {C}")
    
    # 截断数据如果它是 351 (QC版)，我们只需要前 341 做 Signature
    if C > 341:
        print("Note: Truncating data to first 341 channels for figure generation.")
        data_viz = data_4d[..., :341]
    elif C == 341:
        data_viz = data_4d
    else:
        print(f"[WARN] Data channels {C} != 341. Plotting as is.")
        data_viz = data_4d
        
    # 3. 选点
    if VOXEL_COORDS:
        voxel = VOXEL_COORDS
    else:
        voxel = select_voxel(mask)
        
    # 4. 绘图
    plot_figure_1_1(data_viz, voxel, OUTPUT_FIG)

if __name__ == "__main__":
    main()